# Wizualizacja rynku koncertów muzycznych w Polsce

## Część 1: Wczytanie i eksploracja

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_csv('koncerty_polska.csv', parse_dates=['data'])
df['miesiac'] = df['data'].dt.to_period('M').astype(str)
df['wypelnienie'] = df['bilety_sprzedane'] / df['pojemnosc']

print(f'Kształt danych: {df.shape}')
print(f'Unikalne miasta: {df["miasto"].nunique()}')
print(f'Unikalne gatunki: {df["gatunek"].nunique()}')
df.head()

Kształt danych: (1200, 13)
Unikalne miasta: 10
Unikalne gatunki: 10


,event_id,data,miasto,latitude,longitude,gatunek,typ_obiektu,pojemnosc,bilety_sprzedane,cena_biletu_pln,przychod_pln,miesiac,wypelnienie
0,50001,2024-04-12,Poznań,52.4064,16.9252,hip-hop,klub,410,316,170.0,53720,2024-04,0.770732
1,50002,2025-03-11,Gdańsk,54.3520,18.6466,folk,arena,5799,2512,130.0,326560,2025-03,0.433178
2,50003,2024-09-27,Warszawa,52.2297,21.0122,classical,klub,634,414,100.0,41400,2024-09,0.652997
3,50004,2024-04-16,Katowice,50.2649,19.0238,metal,klub,979,735,100.0,73500,2024-04,0.750766
4,50005,2024-03-12,Poznań,52.4064,16.9252,metal,klub,1430,808,80.0,64640,2024-03,0.565035


## Część 2: Wykres słupkowy

In [2]:
def plot_revenue_by_city(df: pd.DataFrame) -> go.Figure:
    df_agg = df.groupby('miasto', as_index=False)['przychod_pln'].sum().sort_values('przychod_pln', ascending=False)
    return px.bar(df_agg, x='miasto', y='przychod_pln', title='Łączny przychód z koncertów wg miast', labels={'miasto': 'Miasto', 'przychod_pln': 'Przychód (PLN)'})

fig2 = plot_revenue_by_city(df)
fig2.show()

## Część 3: Wykresy liniowe

In [3]:
def plot_concerts_per_month(df: pd.DataFrame) -> go.Figure:
    df_agg = df.groupby('miesiac', as_index=False).size()
    return px.line(df_agg, x='miesiac', y='size', title='Liczba koncertów miesięcznie', markers=True, labels={'miesiac': 'Miesiąc', 'size': 'Liczba'})

def plot_concerts_per_month_by_type(df: pd.DataFrame) -> go.Figure:
    df_agg = df.groupby(['miesiac', 'typ_obiektu'], as_index=False).size()
    return px.line(df_agg, x='miesiac', y='size', color='typ_obiektu', title='Koncerty wg obiektu', markers=True, labels={'miesiac': 'Miesiąc', 'size': 'Liczba', 'typ_obiektu': 'Obiekt'})

plot_concerts_per_month(df).show()
plot_concerts_per_month_by_type(df).show()

## Część 4: Histogram i boxplot

In [4]:
def plot_price_histogram(df: pd.DataFrame) -> go.Figure:
    return px.histogram(df, x='cena_biletu_pln', nbins=50, title='Rozkład cen biletów', labels={'cena_biletu_pln': 'Cena (PLN)'})

def plot_revenue_boxplot(df: pd.DataFrame) -> go.Figure:
    return px.box(df, x='typ_obiektu', y='przychod_pln', title='Przychód wg typu obiektu', labels={'typ_obiektu': 'Obiekt', 'przychod_pln': 'Przychód (PLN)'})

plot_price_histogram(df).show()
plot_revenue_boxplot(df).show()

Najwyższe przychody bezsprzecznie generują stadiony oraz festiwale. Wynika to bezpośrednio z faktu, że obiekty te mają zdecydowanie największą pojemność, co gwarantuje największe wpływy z biletów.

## Część 5: Scatter plot

In [5]:
def plot_price_vs_fill(df: pd.DataFrame) -> go.Figure:
    return px.scatter(df, x='cena_biletu_pln', y='wypelnienie', color='gatunek', size='pojemnosc',
                      hover_data=['miasto', 'typ_obiektu'], title='Cena vs Wypełnienie', labels={'cena_biletu_pln': 'Cena (PLN)', 'wypelnienie': 'Wypełnienie'})

plot_price_vs_fill(df).show()

Nie widać żadnej wyraźnej korelacji między ceną biletu a wypełnieniem sali. Nawet najdroższe wydarzenia potrafią zebrać niemal kompletną publiczność.

## Część 6: Mapa

In [6]:
def plot_concerts_map(df: pd.DataFrame) -> go.Figure:
    df_map = df.groupby('miasto', as_index=False).agg(
        srednia_cena=('cena_biletu_pln', 'mean'), liczba_koncertow=('event_id', 'count'),
        laczny_przychod=('przychod_pln', 'sum'), latitude=('latitude', 'first'), longitude=('longitude', 'first'))
    
    return px.scatter_mapbox(df_map, lat='latitude', lon='longitude', size='liczba_koncertow', color='srednia_cena',
                             hover_name='miasto', mapbox_style='open-street-map', zoom=5, title='Mapa miast koncertowych')

plot_concerts_map(df).show()

C:\Users\Maciek\AppData\Local\Temp\ipykernel_7128\3025004263.py:6: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  return px.scatter_mapbox(df_map, lat='latitude', lon='longitude', size='liczba_koncertow', color='srednia_cena',


## Część 7: Subploty 2x2

In [7]:
def plot_dashboard(df: pd.DataFrame) -> go.Figure:
    fig = make_subplots(rows=2, cols=2, subplot_titles=('Przychód wg miast', 'Rozkład cen', 'Liczba koncertów wg gatunku', 'Przychód wg obiektu'))
    
    df_miasta = df.groupby('miasto', as_index=False)['przychod_pln'].sum().sort_values('przychod_pln', ascending=False)
    fig.add_trace(go.Bar(x=df_miasta['miasto'], y=df_miasta['przychod_pln']), row=1, col=1)
    
    fig.add_trace(go.Histogram(x=df['cena_biletu_pln'], nbinsx=50), row=1, col=2)
    
    gatunki = df['gatunek'].value_counts().reset_index()
    fig.add_trace(go.Bar(x=gatunki['gatunek'], y=gatunki['count']), row=2, col=1)
    
    for typ in df['typ_obiektu'].unique():
        fig.add_trace(go.Box(y=df[df['typ_obiektu'] == typ]['przychod_pln'], name=typ), row=2, col=2)
        
    fig.update_layout(title_text='Podsumowanie Rynku Koncertowego w Polsce', height=800, showlegend=False)
    return fig

plot_dashboard(df).show()

## Część 8: Wnioski
1. **Warszawa dominuje rynek -** Organizowana jest tam największa liczba imprez, co przekłada się na największy łączny przychód w kraju.
2. **Najdroższe bilety -** Najwyższe średnie ceny biletów występują na stadionach oraz festiwalach. Kluby pozostają najtańszą opcją.
3. **Brak sezonowości -** Podaż koncertów rozkłada się równomiernie na przestrzeni wszystkich miesięcy. Rynek nie wykazuje drastycznych przerw sezonowych.
4. **Pojemność ponad cenę -** Kluczem do maksymalizacji przychodu jest rozmiar obiektu (stadiony deklasują resztę), a nie windowanie ceny pojedynczego biletu.
5. **Odporność na ceny -** Publiczność chętnie wykupuje miejsca niezależnie od wysokiej ceny biletu (co udowadnia wykres scatter plot).